# Packed Market Model v1 Diagram

Creates a dummy model, prints parameter counts, and writes model artifacts to `artifacts/model_smoke`.

In [ ]:
from pathlib import Path
import torch
from research.mlops.model_artifacts import parameter_summary, write_model_artifacts
from research.packed_market_model.v1.config import ModelConfig
from research.packed_market_model.v1.data import make_dummy_packed_block
from research.packed_market_model.v1.model import PackedMarketModelV1, build_model_mermaid

config = ModelConfig(
    event_feature_names=tuple(f'feature_{i}' for i in range(16)),
    event_feature_dim=16,
    label_names=('future_trade_close', 'future_halt_flag'),
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PackedMarketModelV1(config).to(device)
print(parameter_summary(model))
write_model_artifacts(
    model=model,
    artifact_dir=Path('artifacts/model_smoke'),
    model_config=config,
    input_contract={'events': ['T', config.event_feature_dim], 'origin_positions': ['M']},
    output_contract={'label_predictions': {name: ['M'] for name in config.label_names}},
    architecture_mermaid=build_model_mermaid(),
    summary_notes='Notebook smoke artifact.',
    dummy_input_factory=lambda: ((), make_dummy_packed_block(model_config=config, device=device).x),
)
print('wrote artifacts/model_smoke')